# 四组方法全面对比

汇总 4 种 Anti-CRISPR 蛋白预测方法在**同一测试集**上的完整指标。

| # | 方法 | 论文 | 核心技术 |
|---|------|------|----------|
| 1 | AcRanker | Eitzinger et al., NAR 2020 | XGBoost + 序列 k-mer 特征 |
| 2 | AcrPred | Dao et al., IJBM 2023 | SVM + CTD/PSSM 特征 |
| 3 | DeepAcr | Wandera et al., Mol Cell 2022 | CNN+BiLSTM 纯序列 |
| 4 | **ProteinBERT+PSSM1110 Fusion** | — | ProteinBERT + PSSM1110 融合 |

**指标**：AUC, AUPRC, ACC, SN (Sensitivity), SP (Specificity), F1, MCC, Brier, ECE

**环境**：`lm-hf`（仅读取 JSON 结果文件）

In [1]:
import json, os, glob
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

RESULTS_DIR = '/home/nemophila/projects/protein_bert/Comparison/results'
OUTPUT_DPI = 900

plt.rcParams.update({
    'figure.dpi': OUTPUT_DPI,
    'savefig.dpi': OUTPUT_DPI,
    'figure.facecolor': '#fbf8f3',
    'axes.facecolor': '#fbf8f3',
    'axes.edgecolor': '#4b5563',
    'axes.labelcolor': '#1f2937',
    'axes.titlecolor': '#111827',
    'xtick.color': '#374151',
    'ytick.color': '#374151',
    'grid.color': '#d6d3d1',
    'font.size': 11,
})

METHOD_COLORS = {
    'acranker': '#5B7C99',
    'acrpred': '#C17C5D',
    'deepacr': '#7A9E7E',
    'ours': '#8C6BB1',
}

METRIC_COLORS = {
    'AUC': '#4C78A8',
    'AUPRC': '#6B8E9F',
    'ACC': '#72B7B2',
    'F1': '#54A24B',
    'MCC': '#E45756',
    'SN': '#B279A2',
    'SP': '#F2A65A',
}

# 四组方法的展示名称
METHOD_NAMES = {
    'acranker': 'AcRanker (XGBoost)',
    'acrpred':  'AcrPred (SVM)',
    'deepacr':  'DeepAcr (CNN+BiLSTM)',
    'ours':     'ProteinBERT+PSSM1110 Fusion',
}

## 1. 加载三组对照方法的结果

In [2]:
results = {}

for key in ['acranker', 'acrpred', 'deepacr']:
    path = os.path.join(RESULTS_DIR, f'{key}_metrics.json')
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        results[key] = data['metrics']
        print(f'\u2714 Loaded {key}')
    else:
        print(f'\u2718 Missing {key} — 请先运行对应 notebook')

print(f'\nLoaded {len(results)}/3 baseline results.')

✔ Loaded acranker
✔ Loaded acrpred
✔ Loaded deepacr

Loaded 3/3 baseline results.


## 2. 加载我们的融合模型结果

从 `notebooks/roc_curve_demo.ipynb` 运行后自动保存的 `ours_fusion_metrics.json` 中读取。  
请确保先运行 `roc_curve_demo.ipynb` 生成结果文件。

In [3]:
ours_metrics_path = os.path.join(RESULTS_DIR, 'ours_fusion_metrics.json')

if os.path.exists(ours_metrics_path):
    with open(ours_metrics_path) as f:
        ours_data = json.load(f)
    results['ours'] = ours_data['metrics']
    print('✔ Loaded ProteinBERT+PSSM1110 Fusion metrics from', ours_metrics_path)
    for k, v in results['ours'].items():
        print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')
else:
    raise FileNotFoundError(
        f'{ours_metrics_path} not found.\n'
        'Please run notebooks/roc_curve_demo.ipynb first to generate fusion model results.'
    )

✔ Loaded ProteinBERT+PSSM1110 Fusion metrics from /home/nemophila/projects/protein_bert/Comparison/results/ours_fusion_metrics.json
  AUC: 0.9302
  AUPRC: 0.7175
  F1: 0.6522
  MCC: 0.6287
  Brier: 0.0464
  ECE: 0.0495
  ACC: 0.9441
  SN: 0.5769
  SP: 0.9808
  Threshold: 0.2500


## 3. 生成对比表

In [4]:
# 指标展示顺序
METRIC_ORDER = ['AUC', 'AUPRC', 'ACC', 'SN', 'SP', 'F1', 'MCC', 'Brier', 'ECE']

rows = {}
for key, name in METHOD_NAMES.items():
    if key in results and results[key] is not None:
        row = {}
        for m in METRIC_ORDER:
            v = results[key].get(m)
            if v is not None:
                row[m] = round(v, 4)
            else:
                row[m] = '—'
        rows[name] = row

comparison_df = pd.DataFrame(rows).T
comparison_df.index.name = 'Method'

# 高亮：对于每个指标，标出最优值
# AUC, AUPRC, ACC, SN, SP, F1, MCC → 越大越好; Brier, ECE → 越小越好
print('=' * 80)
print('Anti-CRISPR Protein Prediction — Four-way Comparison')
print('=' * 80)
comparison_df

Anti-CRISPR Protein Prediction — Four-way Comparison


,AUC,AUPRC,ACC,SN,SP,F1,MCC,Brier,ECE
Method,,,,,,,,,
AcRanker (XGBoost),0.8374,0.3904,0.8322,0.5000,0.8654,0.3514,0.2811,0.0731,0.0555
AcrPred (SVM),0.9430,0.6862,0.9266,0.4615,0.9731,0.5333,0.5017,0.1164,0.2398
DeepAcr (CNN+BiLSTM),0.8070,0.4235,0.8182,0.5385,0.8462,0.3500,0.2825,0.0801,0.1055
ProteinBERT+PSSM1110 Fusion,0.9302,0.7175,0.9441,0.5769,0.9808,0.6522,0.6287,0.0464,0.0495


## 4. 高亮最优结果

In [5]:
higher_better = {'AUC', 'AUPRC', 'ACC', 'SN', 'SP', 'F1', 'MCC'}
lower_better  = {'Brier', 'ECE'}


def highlight_best(df):
    """返回带样式的 DataFrame，最优值加粗高亮。"""
    styled = df.copy().astype(str)
    for col in df.columns:
        numeric_vals = pd.to_numeric(df[col], errors='coerce')
        valid = numeric_vals.dropna()
        if valid.empty:
            continue
        if col in higher_better:
            best_idx = valid.idxmax()
        elif col in lower_better:
            best_idx = valid.idxmin()
        else:
            continue
        styled.at[best_idx, col] = f'**{df.at[best_idx, col]}**'
    return styled


highlighted = highlight_best(comparison_df)
highlighted

,AUC,AUPRC,ACC,SN,SP,F1,MCC,Brier,ECE
Method,,,,,,,,,
AcRanker (XGBoost),0.8374,0.3904,0.8322,0.5,0.8654,0.3514,0.2811,0.0731,0.0555
AcrPred (SVM),**0.943**,0.6862,0.9266,0.4615,0.9731,0.5333,0.5017,0.1164,0.2398
DeepAcr (CNN+BiLSTM),0.807,0.4235,0.8182,0.5385,0.8462,0.35,0.2825,0.0801,0.1055
ProteinBERT+PSSM1110 Fusion,0.9302,**0.7175**,**0.9441**,**0.5769**,**0.9808**,**0.6522**,**0.6287**,**0.0464**,**0.0495**


## 5. 保存对比表为 CSV（方便论文直接引用）

In [6]:
out_csv = os.path.join(RESULTS_DIR, 'four_way_comparison.csv')
comparison_df.to_csv(out_csv)
print(f'Comparison table saved to {out_csv}')

Comparison table saved to /home/nemophila/projects/protein_bert/Comparison/results/four_way_comparison.csv


## 6. 可视化：雷达图对比

In [7]:
radar_metrics = ['AUC', 'AUPRC', 'ACC', 'SN', 'SP', 'F1', 'MCC']

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
angles += angles[:1]

for key, name in METHOD_NAMES.items():
    if key not in results or results[key] is None:
        continue
    vals = []
    for m in radar_metrics:
        v = results[key].get(m)
        vals.append(v if v is not None and not isinstance(v, str) else 0)
    vals += vals[:1]
    color = METHOD_COLORS[key]
    ax.plot(angles, vals, 'o-', label=name, color=color, linewidth=2.2, markersize=4)
    ax.fill(angles, vals, alpha=0.08, color=color)

ax.set_thetagrids(np.degrees(angles[:-1]), radar_metrics)
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.55, linestyle='--', linewidth=0.8)
ax.spines['polar'].set_color('#94a3b8')
ax.spines['polar'].set_linewidth(1.0)
ax.legend(loc='upper right', bbox_to_anchor=(1.38, 1.12), fontsize=9, frameon=True)
ax.set_title('Anti-CRISPR Prediction — Method Comparison', y=1.10, fontsize=13, fontweight='semibold')
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'radar_comparison.png'), dpi=OUTPUT_DPI, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'Radar chart saved at {OUTPUT_DPI} dpi.')

Radar chart saved at 900 dpi.


## 7. 可视化：柱状图对比

In [8]:
bar_metrics = ['AUC', 'AUPRC', 'ACC', 'SN', 'SP', 'F1', 'MCC']
method_labels = []
bar_data = {m: [] for m in bar_metrics}

for key, name in METHOD_NAMES.items():
    if key not in results or results[key] is None:
        continue
    method_labels.append(name.split('(')[0].strip())
    for m in bar_metrics:
        v = results[key].get(m)
        bar_data[m].append(v if v is not None and not isinstance(v, str) else 0)

x = np.arange(len(method_labels))
width = 0.11  # 7 metrics: AUC, AUPRC, ACC, SN, SP, F1, MCC
fig, ax = plt.subplots(figsize=(12, 5.2))

for i, m in enumerate(bar_metrics):
    ax.bar(
        x + i * width,
        bar_data[m],
        width,
        label=m,
        color=METRIC_COLORS[m],
        edgecolor='#f8fafc',
        linewidth=0.8,
    )

ax.set_xlabel('Method')
ax.set_ylabel('Score')
ax.set_title('Anti-CRISPR Prediction — Key Metrics Comparison', fontweight='semibold')
ax.set_xticks(x + width * (len(bar_metrics) - 1) / 2)
ax.set_xticklabels(method_labels, rotation=15, ha='right')
ax.set_ylim(0, 1.1)
ax.yaxis.grid(True, linestyle='--', alpha=0.45)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(frameon=True, ncol=1, loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=8)
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'bar_comparison.png'), dpi=OUTPUT_DPI, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'Bar chart saved at {OUTPUT_DPI} dpi.')

Bar chart saved at 900 dpi.


## 8. 可视化：ROC 曲线对比

从各方法保存的 `*_predictions.npz` 文件中加载 `y_true` 和 `y_prob`，绘制四方法 ROC 曲线。

In [9]:
from sklearn.metrics import roc_curve, roc_auc_score

pred_files = {
    'acranker': 'acranker_predictions.npz',
    'acrpred':  'acrpred_predictions.npz',
    'deepacr':  'deepacr_predictions.npz',
    'ours':     'ours_predictions.npz',
}

fig, ax = plt.subplots(figsize=(7, 6))

for key, fname in pred_files.items():
    fpath = os.path.join(RESULTS_DIR, fname)
    if not os.path.exists(fpath):
        print(f'⚠ Missing {fpath}, skipping {key}')
        continue
    data = np.load(fpath)
    y_true, y_prob = data['y_true'], data['y_prob']
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val = roc_auc_score(y_true, y_prob)
    ax.plot(fpr, tpr,
            label=f'{METHOD_NAMES[key]} (AUC={auc_val:.3f})',
            color=METHOD_COLORS[key], linewidth=2.2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, linewidth=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Anti-CRISPR Prediction — ROC Curve Comparison',
             fontweight='semibold')
ax.legend(loc='lower right', fontsize=9, frameon=True)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.05)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, linestyle='--', alpha=0.35)
ax.set_axisbelow(True)
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'roc_comparison.png'),
            dpi=OUTPUT_DPI, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'ROC comparison chart saved at {OUTPUT_DPI} dpi.')

ROC comparison chart saved at 900 dpi.
